### Digital Twin Pseudo Data Code Statistical Measurements

#### Keith R. Holdaway FGS

#### Digital Twin Setup (Pseudocode):

In [1]:
import numpy as np
import pandas as pd

# Define ranges for rock properties
n_samples = 50000
porosity = np.random.uniform(0.05, 0.35, n_samples)
clay_volume = np.random.uniform(0, 0.5, n_samples)
# ... other properties ...

# Use a rock physics model (e.g., Coates-Denoo formula) to calculate PERMEABILITY (our target)
permeability = (100 * porosity**2) / (1 - porosity) # Simplified model

# Use forward models to calculate LOG RESPONSES (our features)
density_log = 2.65 - porosity * 1.0 + clay_volume * 0.3 # g/cc
neutron_log = porosity * 1.0 + clay_volume * 0.4 # limestone units
# ... calculate other logs ...

# Create a DataFrame
synthetic_data = pd.DataFrame({
    'Density': density_log,
    'Neutron': neutron_log,
    'Porosity': porosity,
    'Vclay': clay_volume,
    'Permeability': permeability # This is our known, perfect target
})

#### AI Model Training (Python with Scikit-learn/TensorFlow):

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Features (logs) and Target (Permeability)
X = synthetic_data[['Density', 'Neutron', 'Vclay']]
y = synthetic_data['Permeability']

# Split the synthetic data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Create and train the model
mlp_model = MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000)
mlp_model.fit(X_train, y_train)

# Evaluate on the synthetic test set
y_pred = mlp_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error on Synthetic Data: {mse}")

Mean Squared Error on Synthetic Data: 0.0008785861593137914


#### Transfer to Real Data:

Once the model performs well on the synthetic data (with added noise and complexities), the petrophysicist can then fine-tune it on the limited real core and log data they have. This transfer learning approach is much more effective than starting from scratch.



In [3]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Assuming you have your test set predictions and true values
# y_test = true values from your digital twin
# y_pred = predictions from your AI model

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Calculate the range of the true target values
target_range = y_test.max() - y_test.min()
target_mean = y_test.mean()
target_std = y_test.std()

print("=== Model Performance Evaluation ===")
print(f"Mean Squared Error (MSE): {mse:.6f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.6f}")
print(f"Mean Absolute Error (MAE): {mae:.6f}")
print(f"R² Score: {r2:.6f}")
print("\n=== Context of Target Variable ===")
print(f"Target Variable Range: {target_range:.4f}")
print(f"Target Variable Mean: {target_mean:.4f}")
print(f"Target Variable Std Dev: {target_std:.4f}")
print("\n=== Relative Error Metrics ===")
print(f"RMSE / Range of Target: {(rmse / target_range) * 100:.2f}%")
print(f"RMSE / Mean of Target: {(rmse / target_mean) * 100:.2f}%")

=== Model Performance Evaluation ===
Mean Squared Error (MSE): 0.000879
Root Mean Squared Error (RMSE): 0.029641
Mean Absolute Error (MAE): 0.021747
R² Score: 0.999970

=== Context of Target Variable ===
Target Variable Range: 18.5818
Target Variable Mean: 6.6107
Target Variable Std Dev: 5.4079

=== Relative Error Metrics ===
RMSE / Range of Target: 0.16%
RMSE / Mean of Target: 0.45%


In [4]:
# Calculate baseline RMSE (predicting the mean for every sample)
y_baseline = np.full_like(y_test, fill_value=y_test.mean())
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_baseline))
print(f"Baseline RMSE (always predict mean): {baseline_rmse:.6f}")
print(f"Your model's RMSE is {((baseline_rmse - rmse) / baseline_rmse) * 100:.1f}% better than the baseline.")

Baseline RMSE (always predict mean): 5.407652
Your model's RMSE is 99.5% better than the baseline.


#### Conclusion
An MSE of 0.00154 is likely a very good result, especially if you are predicting a normalized or logged property like log(Permeability) or a scaled porosity. However, you must calculate the R² score and compare the RMSE to the scale of your data to be certain.

This process of contextualizing your error metrics is a fundamental skill in machine learning. The fact that you're asking this question shows you're thinking like a true data scientist!